In [1]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

In [2]:
training_data = datasets.FashionMNIST(
    root='data',
    train=True,
    download=True,
    transform=ToTensor(),
)

test_data = datasets.FashionMNIST(
    root='data',
    train=False,
    download=True,
    transform=ToTensor(),
)

100%|██████████| 26.4M/26.4M [00:01<00:00, 13.3MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 209kB/s]
100%|██████████| 4.42M/4.42M [00:01<00:00, 3.90MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 26.1MB/s]


In [3]:
batch_size = 64

train_dataloader = DataLoader(training_data, batch_size=batch_size)
test_dataloader = DataLoader(test_data, batch_size=batch_size)

for X, y in test_dataloader:
    print(f'shape of X [N, C, H, W]: {X.shape}')
    print(f'shape of y: {y.shape} {y.dtype}')
    break

shape of X [N, C, H, W]: torch.Size([64, 1, 28, 28])
shape of y: torch.Size([64]) torch.int64


In [19]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else 'cpu'
print(f'Using {device} device')

class NeuralNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.liner_relu_stack = nn.Sequential(
            nn.Linear(28*28, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 10),   
        )
    
        
    def forward(self, x):
        x = self.flatten(x)
        output = self.liner_relu_stack(x)
        return output
    
model = NeuralNetwork().to(device)
print(model)

Using cuda device
NeuralNetwork(
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (liner_relu_stack): Sequential(
    (0): Linear(in_features=784, out_features=512, bias=True)
    (1): ReLU()
    (2): Linear(in_features=512, out_features=512, bias=True)
    (3): ReLU()
    (4): Linear(in_features=512, out_features=10, bias=True)
  )
)


In [9]:
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=1e-3)


In [18]:
def train(dataloader, model, loss_fn, optimizer):
    size = len(dataloader.dataset)
    model.train()
    for batch, (X, y) in enumerate(dataloader):
        X, y = X.to(device), y.to(device)
        
        pred = model(X)
        loss = loss_fn(pred, y)
        
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
        
        if batch % 100 == 0:
            loss, current = loss.item(), (batch + 1) * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

In [11]:
def test(dataloader, mode, loss_fn, optimizer):
    size = len(dataloader.dataset)
    num_batches = len(dataloader)
    model.eval()
    test_loss, correct = 0, 0
    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.to(device), y.to(device)
            pred = model(X)
            test_loss += loss_fn(pred, y ).item()
            correct += (pred.argmax(1) == y).type(torch.float).sum().item()
    test_loss /= num_batches
    correct /= size
    print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [20]:
epchos = 5
for t in range(epchos):
    print(f'epcho {t+1}')
    train(train_dataloader, model, loss_fn, optimizer)
    test(test_dataloader, model, loss_fn, optimizer)
print('done')

epcho 1
loss: 2.299913  [   64/60000]
loss: 2.302519  [ 6464/60000]
loss: 2.315415  [12864/60000]
loss: 2.315984  [19264/60000]
loss: 2.299352  [25664/60000]
loss: 2.322520  [32064/60000]
loss: 2.314844  [38464/60000]
loss: 2.316183  [44864/60000]
loss: 2.302334  [51264/60000]
loss: 2.323059  [57664/60000]
Test Error: 
 Accuracy: 12.7%, Avg loss: 2.312615 

epcho 2
loss: 2.299913  [   64/60000]
loss: 2.302519  [ 6464/60000]
loss: 2.315415  [12864/60000]
loss: 2.315984  [19264/60000]
loss: 2.299352  [25664/60000]
loss: 2.322520  [32064/60000]
loss: 2.314844  [38464/60000]
loss: 2.316183  [44864/60000]
loss: 2.302334  [51264/60000]
loss: 2.323059  [57664/60000]
Test Error: 
 Accuracy: 12.7%, Avg loss: 2.312615 

epcho 3
loss: 2.299913  [   64/60000]
loss: 2.302519  [ 6464/60000]
loss: 2.315415  [12864/60000]
loss: 2.315984  [19264/60000]
loss: 2.299352  [25664/60000]
loss: 2.322520  [32064/60000]
loss: 2.314844  [38464/60000]
loss: 2.316183  [44864/60000]
loss: 2.302334  [51264/60000]
lo